In [3]:
#importing the dataset
import numpy as np
import pandas as pd
# names is used to define column name like label and message or sep is used to define seperater
messages = pd.read_csv('SMSSpamcollection', sep = '\t', names=["label","message"])

In [4]:
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [5]:
messages.shape

(5572, 2)

In [6]:
# find sentence in that location , 100 is number of rows
messages['message'].loc[100]

"Please don't text me anymore. I have nothing else to say."

In [7]:
#Data cleaning and preprocessing
import re  # Import regex module for text cleaning (removing symbols, numbers, etc.)
# nltk is lib.
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sumit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [9]:
corpus =[] # Empty list to store processed (cleaned + stemmed) sentences
for i in range(0, len(messages)): # Loop through each message (row) in dataset
    review = re.sub('[^a-zA-Z]',' ',messages['message'][i]) # Replace everything except alphabets with space
    # Example: "Hello!!! 123" → "Hello     "
    review = review.lower() # Convert entire text to lowercase
    review = review.split() # Convert sentence into list of words (tokenization)
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]  # 1. Remove stopwords (like 'the', 'is', 'and')
    # 2. Apply stemming (running → run, playing → play)
    review = ' '.join(review) # Join words back into a single string
    # Example: ["run", "fast"] → "run fast"
    corpus.append(review)   # Add cleaned sentence into corpus list

In [10]:
corpus

['go jurong point crazi avail bugi n great world la e buffet cine got amor wat',
 'ok lar joke wif u oni',
 'free entri wkli comp win fa cup final tkt st may text fa receiv entri question std txt rate c appli',
 'u dun say earli hor u c alreadi say',
 'nah think goe usf live around though',
 'freemsg hey darl week word back like fun still tb ok xxx std chg send rcv',
 'even brother like speak treat like aid patent',
 'per request mell mell oru minnaminungint nurungu vettam set callertun caller press copi friend callertun',
 'winner valu network custom select receivea prize reward claim call claim code kl valid hour',
 'mobil month u r entitl updat latest colour mobil camera free call mobil updat co free',
 'gonna home soon want talk stuff anymor tonight k cri enough today',
 'six chanc win cash pound txt csh send cost p day day tsandc appli repli hl info',
 'urgent week free membership prize jackpot txt word claim c www dbuk net lccltd pobox ldnw rw',
 'search right word thank breather

In [11]:
# creating the Bag of Words model

In [12]:

from sklearn.feature_extraction.text import CountVectorizer
# Import CountVectorizer → used to convert text (corpus) into numerical features (Bag of Words)

cv = CountVectorizer(max_features = 2500, binary= True)
# Create object of CountVectorizer with parameters:
# max_features=2500 → keep only top 2500 most frequent words (important for performance)
# binary=True → use 0 or 1 instead of word count
#                1 = word present, 0 = word absent

X = cv.fit_transform(corpus).toarray()
# Step 1: fit_transform(corpus)
#   - fit → learn vocabulary (unique words from corpus)
#   - transform → convert each sentence into vector form

# Step 2: .toarray()
#   - convert sparse matrix into normal 2D array (matrix form)

# Final result:
# X = numerical representation of text data

In [13]:
#X[302]
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [14]:
# Convert categorical labels (like 'spam' and 'ham') into dummy/one-hot encoded variables
y = pd.get_dummies(messages['label'])
# Select only the second column (index 1), which usually represents one class (e.g., 'spam')
# This avoids the dummy variable trap (removing redundancy)
y=y.iloc[:,1].values

In [15]:
y

array([False, False,  True, ..., False, False, False])

In [16]:
# Train Test Split
# Import the function used to split dataset into training and testing sets
from sklearn.model_selection import train_test_split
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X,  # Features (independent variables, e.g., text vectors from CountVectorizer)
                                                    y,  # Labels (dependent variable, e.g., spam = 1, ham = 0)
                                                    test_size = 0.20,  # 20% data will be used for testing, 80% for training
                                                    random_state = 0)  # Fixes randomness so results are reproducible every time

In [17]:
X_train, y_train

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 array([False, False, False, ...,  True, False, False]))

In [18]:
# Import the Naive Bayes model (used for classification, especially text data)
from sklearn.naive_bayes import MultinomialNB
# Create the model and train it using training data
spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [19]:
#prediction
# Use the trained model to predict labels for test data
# X_test → unseen feature vectors (same format as X_train, e.g., TF-IDF)
# .predict() → applies learned patterns to classify each input
# Output → predicted labels (e.g., spam = 1, not spam = 0)
y_pred=spam_detect_model.predict(X_test)

In [20]:
# Import evaluation metrics from sklearn
# accuracy_score → gives overall correctness of model (0 to 1)
# classification_report → gives detailed metrics:
#   - Precision (how many predicted positives are correct)
#   - Recall (how many actual positives are captured)
#   - F1-score (balance between precision & recall)
#   - Support (number of actual samples per class)
from sklearn.metrics import accuracy_score, classification_report

In [21]:
# Compare actual labels (y_test) with predicted labels (y_pred)
# accuracy_score → calculates how many predictions are correct
# Formula: (correct predictions / total predictions)
score=accuracy_score(y_test,y_pred)
print(score)

0.9856502242152466


In [22]:
# Import evaluation metrics from sklearn
# Compare actual labels with predicted labels
# y_test → true/actual values
# y_pred → model predictions
# classification_report → shows precision, recall, f1-score, and support
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.99      0.99      0.99       955
        True       0.97      0.93      0.95       160

    accuracy                           0.99      1115
   macro avg       0.98      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [23]:
# creating the TFIDF model

In [24]:
# Import TF-IDF Vectorizer from sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
# Create TF-IDF object
# max_features=2500 → keep only top 2500 most important words
tv = TfidfVectorizer(max_features=2500,ngram_range=(1,2))
# Learn vocabulary + IDF values from corpus (fit)
# Convert text data into numerical feature vectors (transform)
# toarray() → convert sparse matrix into normal array
X=tv.fit_transform(corpus).toarray()

In [25]:
#Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state = 0)

In [26]:
# Import Multinomial Naive Bayes model from sklearn
from sklearn.naive_bayes import MultinomialNB
# Create the model and train it using training data
# X_train → feature vectors (e.g., TF-IDF or CountVectorizer output)
# y_train → target labels (e.g., spam = 1, not spam = 0)
# .fit() → trains the model by learning patterns from the data
spam_detect_model = MultinomialNB().fit(X_train,y_train)

In [27]:
# Prediction
# Use the trained model to predict labels for test data
# X_test → unseen feature vectors (same format as X_train, e.g., TF-IDF)
# .predict() → applies learned patterns to classify each input
# Output → predicted labels (e.g., spam = 1, not spam = 0)
y_pred = spam_detect_model.predict(X_test)

In [28]:
# Compare actual labels (y_test) with predicted labels (y_pred)
# accuracy_score → calculates how many predictions are correct
# Formula: (correct predictions / total predictions)
score=accuracy_score(y_test,y_pred)
print(score)

0.97847533632287


In [29]:
# Import classification_report
# Print detailed evaluation of your model
# y_test → actual (true) labels
# y_pred → predicted labels by the model
# classification_report → shows precision, recall, f1-score, support for each class
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.98      1.00      0.99       955
        True       1.00      0.85      0.92       160

    accuracy                           0.98      1115
   macro avg       0.99      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [30]:
# Import Random Forest classifier from sklearn
from sklearn.ensemble import RandomForestClassifier
# Create the model
# RandomForest → ensemble model (many decision trees combined)
classifier=RandomForestClassifier()
# Train the model using training data
# X_train → feature vectors (e.g., TF-IDF output)
# y_train → actual labels (spam / not spam)
# .fit() → model learns patterns from data
classifier.fit(X_train,y_train)

RandomForestClassifier()

In [31]:
X_train,y_train

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]),
 array([False, False, False, ...,  True, False, False]))

In [32]:
y_pred=classifier.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9847533632286996
              precision    recall  f1-score   support

       False       0.98      1.00      0.99       955
        True       1.00      0.89      0.94       160

    accuracy                           0.98      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.98      0.98      1115



In [33]:
# prepare data which use to train word2vec model from scratch

In [34]:
# Import WordNet Lemmatizer from nltk (Natural Language Toolkit)
from nltk.stem import WordNetLemmatizer
# Create lemmatizer object
# This will be used to convert words into their base (dictionary) form
lemmatizer=WordNetLemmatizer()

In [35]:
# Create an empty list to store processed text
corpus=[]
# Loop through all messages
for i in range(0, len(messages)):
     # Remove all non-alphabet characters (keep only a-z, A-Z)
    review = re.sub('[^a-zA-Z]',' ',messages['message'][i])
    review = review.lower()
    # Split sentence into words (tokenization)
    review = review.split()
    # Lemmatize each word (convert to root form)
    # Also remove stopwords like "the", "is", "and"
    review = [lemmatizer.lemmatize(word) for word in review if not word in stopwords.words('english')]
    # Join words back into a sentence
    review = ' '.join(review)
    # Add cleaned sentence to corpus
    corpus.append(review)

In [36]:
# Import sentence tokenizer from nltk
# sent_tokenize → splits a paragraph into sentences
from nltk import sent_tokenize
# Import simple_preprocess from gensim
# simple_preprocess → cleans text + splits into words (tokens)
# - removes punctuation
# - converts to lowercase
# - keeps only meaningful words
from gensim.utils import simple_preprocess

In [37]:
corpus

['go jurong point crazy available bugis n great world la e buffet cine got amore wat',
 'ok lar joking wif u oni',
 'free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply',
 'u dun say early hor u c already say',
 'nah think go usf life around though',
 'freemsg hey darling week word back like fun still tb ok xxx std chgs send rcv',
 'even brother like speak treat like aid patent',
 'per request melle melle oru minnaminunginte nurungu vettam set callertune caller press copy friend callertune',
 'winner valued network customer selected receivea prize reward claim call claim code kl valid hour',
 'mobile month u r entitled update latest colour mobile camera free call mobile update co free',
 'gonna home soon want talk stuff anymore tonight k cried enough today',
 'six chance win cash pound txt csh send cost p day day tsandcs apply reply hl info',
 'urgent week free membership prize jackpot txt word claim c www dbuk net lccltd pobox ldnw rw'

In [38]:
# Create empty list to store tokenized words
words=[]
# Loop through each cleaned sentence in corpus
for sent in corpus:
    # Split paragraph/sentence into smaller sentences
    # sent_tokenize → ["sentence1", "sentence2"]
    sent_token = sent_tokenize(sent)
    # Loop through each small sentence
    for sent in sent_token:
        # Convert sentence → list of clean words (tokens)
        # simple_preprocess:
        # - convert a document into a list of lowercase token
        # - remove punctuation
        # - keep only useful words
        # Example: "AI is Awesome!" → ['ai', 'awesome']
        words.append(simple_preprocess(sent))

In [39]:
words

[['go',
  'jurong',
  'point',
  'crazy',
  'available',
  'bugis',
  'great',
  'world',
  'la',
  'buffet',
  'cine',
  'got',
  'amore',
  'wat'],
 ['ok', 'lar', 'joking', 'wif', 'oni'],
 ['free',
  'entry',
  'wkly',
  'comp',
  'win',
  'fa',
  'cup',
  'final',
  'tkts',
  'st',
  'may',
  'text',
  'fa',
  'receive',
  'entry',
  'question',
  'std',
  'txt',
  'rate',
  'apply'],
 ['dun', 'say', 'early', 'hor', 'already', 'say'],
 ['nah', 'think', 'go', 'usf', 'life', 'around', 'though'],
 ['freemsg',
  'hey',
  'darling',
  'week',
  'word',
  'back',
  'like',
  'fun',
  'still',
  'tb',
  'ok',
  'xxx',
  'std',
  'chgs',
  'send',
  'rcv'],
 ['even', 'brother', 'like', 'speak', 'treat', 'like', 'aid', 'patent'],
 ['per',
  'request',
  'melle',
  'melle',
  'oru',
  'minnaminunginte',
  'nurungu',
  'vettam',
  'set',
  'callertune',
  'caller',
  'press',
  'copy',
  'friend',
  'callertune'],
 ['winner',
  'valued',
  'network',
  'customer',
  'selected',
  'receivea',
 

In [40]:
import gensim

In [41]:
# Lets train Word2vec from scratch
#gensim library: used for Word2vec, Doc2vec,topic modeling
#words: your input data
#window=5: context window size, looks at 5 words before & after target word
#min_count=2: ignore words that appear less than 2 times
model = gensim.models.Word2Vec(words,window=5,min_count=2)

In [42]:
# Get list of all words (vocabulary) learned by the Word2Vec model
model.wv.index_to_key

['call',
 'get',
 'ur',
 'gt',
 'lt',
 'go',
 'day',
 'ok',
 'free',
 'know',
 'come',
 'like',
 'good',
 'time',
 'got',
 'love',
 'text',
 'want',
 'send',
 'one',
 'need',
 'txt',
 'today',
 'going',
 'stop',
 'home',
 'lor',
 'sorry',
 'see',
 'still',
 'mobile',
 'take',
 'back',
 'da',
 'reply',
 'dont',
 'think',
 'tell',
 'week',
 'phone',
 'hi',
 'new',
 'later',
 'please',
 'pls',
 'co',
 'msg',
 'min',
 'dear',
 'night',
 'make',
 'message',
 'well',
 'say',
 'thing',
 'much',
 'hope',
 'oh',
 'claim',
 'great',
 'hey',
 'number',
 'give',
 'happy',
 'work',
 'friend',
 'wat',
 'way',
 'yes',
 'www',
 'let',
 'prize',
 'right',
 'tomorrow',
 'already',
 'tone',
 'said',
 'ask',
 'win',
 'amp',
 'cash',
 'life',
 'im',
 'yeah',
 'really',
 'babe',
 'meet',
 'find',
 'miss',
 'morning',
 'last',
 'service',
 'year',
 'uk',
 'thanks',
 'care',
 'would',
 'anything',
 'com',
 'also',
 'nokia',
 'lol',
 'every',
 'feel',
 'keep',
 'pick',
 'sure',
 'sent',
 'contact',
 'urgent',


In [43]:
#corpus_count → number of sentences (documents) used during training
model.corpus_count

5564

In [44]:
# Number of times the model sees the entire dataset
model.epochs

5

In [45]:
# Find words similar to 'kid' based on learned word embeddings
model.wv.similar_by_word('prize')

[('claim', 0.9993193745613098),
 ('call', 0.9992280602455139),
 ('line', 0.9992009401321411),
 ('cash', 0.9991658329963684),
 ('land', 0.9989680051803589),
 ('guaranteed', 0.9989228844642639),
 ('show', 0.9989190697669983),
 ('free', 0.9989120364189148),
 ('mobile', 0.998862624168396),
 ('awarded', 0.9988545775413513)]

In [46]:
model.wv['kid'].shape

(100,)

In [75]:
def avg_word2vec(doc):
    word_vectors = [
        model.wv[word]
        for word in doc
        if word in model.wv.index_to_key
    ]

    # If no valid word found
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    # Return average vector
    return np.mean(word_vectors, axis=0)

In [76]:
!pip install tqdm

In [77]:
from tqdm import tqdm

In [78]:
words[73]

['performed']

In [79]:
type(model.wv.index_to_key)

list

In [80]:
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

100%|██████████| 5564/5564 [00:00<00:00, 5856.76it/s]


In [83]:
type(X)

list

In [85]:
X_new = np.array(X)

In [87]:
X_new[3]

array([-1.21352516e-01,  3.10077339e-01,  1.36373295e-02,  1.14580393e-02,
        3.40673774e-02, -4.99477297e-01,  9.89740565e-02,  6.09943330e-01,
       -1.22254245e-01, -2.03083798e-01, -1.87044010e-01, -5.12312353e-01,
       -1.00600779e-01,  1.75180852e-01,  5.78054525e-02, -1.86539546e-01,
       -2.51985993e-03, -4.02275175e-01, -3.48552801e-02, -5.15654325e-01,
        1.88526079e-01,  2.16957331e-01,  1.90207884e-01, -2.49594137e-01,
       -1.15150988e-01, -4.50339504e-02, -3.35051417e-01, -2.50253528e-01,
       -2.71900892e-01,  9.98589993e-02,  3.93966794e-01,  2.47744117e-02,
        6.48087859e-02, -1.95697069e-01, -1.82185903e-01,  3.86712521e-01,
        3.37929279e-02, -1.98174104e-01, -2.00029328e-01, -5.85427940e-01,
        4.86995131e-02, -2.30472803e-01, -9.76802930e-02,  9.51688271e-04,
        3.07929665e-01, -1.00463033e-01, -2.16528535e-01, -5.20950668e-02,
        1.15996279e-01,  1.28961742e-01,  2.25092784e-01, -3.48702699e-01,
       -8.68720412e-02, -

In [88]:
X_new

array([[-0.09743723,  0.25634247,  0.01189643, ..., -0.28465286,
         0.12398931,  0.02713802],
       [-0.05892743,  0.17189424,  0.00773344, ..., -0.19110325,
         0.08467674,  0.01943377],
       [-0.0919248 ,  0.24665232,  0.01724054, ..., -0.26679218,
         0.11573531,  0.02503798],
       ...,
       [-0.01375738,  0.03747864, -0.00407798, ..., -0.04409155,
         0.0239525 ,  0.00384347],
       [-0.10188523,  0.2723484 ,  0.012904  , ..., -0.30129063,
         0.13061461,  0.03255282],
       [-0.07873511,  0.19673085,  0.00195613, ..., -0.21674938,
         0.08826499,  0.02170736]])